In [ ]:
import os 
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
from pandas import Index
from typing import Any, TypedDict

In [ ]:
%matplotlib notebook

In [ ]:
""" 
Required classes. 
"""

class SummaryMetricInfo(TypedDict):
    values: np.ndarray
    save_path: Path
    header: list[str]

In [ ]:
"""
Interactive analysis functionality.
"""

# def show_col_summary(column: Any, frame):
#     print(frame[column].describe())

# def show_summary(dfs: list[pd.DataFrame], v_dir: Path):
#     print(f"Results aggregated over {v_dir.stem} in {v_dir.parent.parent.stem}/{v_dir.parent.stem}")
#     for frame in dfs:
#         interact(show_col_summary, frame=frame, column=list(frame.columns))
#         frame 


In [ ]:
""" 
Visualization functionality. 
"""

def viz_sep_col(metric: SummaryMetricInfo, save_fig: bool = True, save_derived_metric: bool = False) -> None:
    """
    For a metric object instance, visualize each col of `values` with its associated string in `header`.
    Internal output is figure with plots per column, header label combo.
    Optionally save each to compressed numpy file with flags (NOT recommended).
    Optionally save figure (Recommended). 
    TODO: But with what path, might as well maybe create separate instance separate from global df, not really. 
    """
    n_cols = metric["values"].shape[1]
    fig, axs = plt.subplots(1, n_cols, sharex=True, sharey=True, figsize=(4 * n_cols, 4))
    fig.subplots_adjust(wspace=0, hspace=0)

    for i, _ in enumerate(metric["header"]):
        col_vals = metric["values"][:,i]
        col_name = metric["header"][i]
        col_path = Path(metric['save_path'].resolve() / metric["header"][i])
        axs[i].plot(
            col_vals, 
            linestyle="dashed", 
            color="black", 
            marker="o", 
            label=col_name
        )
        axs[i].legend()

        if save_derived_metric:
            np.savez_compressed(file=col_path, args=col_vals)

    if save_fig: 
        fig.savefig(metric['save_path'].resolve(), format="png")

def viz_all_col(metric: SummaryMetricInfo, save_fig: bool = True, save_derived_metric: bool = True) -> None:
    """
    For a metric object instance, visualize all col of `values` with its associated string in `header`.
    Internal output is figure with 1 plot for all column , labelled by col name in Index.
    Optionally save as compressed numpy file with flags per viz (Recommended).
    Optionally save figure (Recommended) per viz. 
    """
    fig, axs = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(4, 4))
    metric_df = pd.DataFrame(metric["values"], columns=metric['header'])
    metric_df.plot(sharex=True, sharey=True, ax=axs)

    if save_derived_metric:
        metric_df.to_csv(metric['save_path'].resolve(), index=False)

    if save_fig: 
        fig.savefig(metric['save_path'].resolve(), format="png")

In [ ]:
""" 
GCS analysis helpers. 
"""

def stitch_cs_metrics(version_dir: Path, cs_type: str = "GCS", post_stitch_delete: bool = False) -> None:
    """
    Stitches together specified epoch correlation store metrics per run.
    Result is saved to run folder. 
    `post_stitch_delete` flag can be used to delete after stitching is done for a run.
    """
    # Get GCS folders sorted by run they are in
    gcs_dirs = sorted(version_dir.rglob(f"run*/train/{cs_type}*/", case_sensitive=False))

    for dir in gcs_dirs:
        # Get all gcs metrics in epoch directory
        metrics_paths = sorted(dir.rglob("*", case_sensitive=False))
        metrics_dfs = map(lambda m: pd.read_csv(m), metrics_paths) 
        run_metrics = pd.concat(metrics_dfs, axis=0)
        # Note navigation: We are in the directory at same level as curr file paths
        # In current run there is only 1 GCS, so no highly unique name needed much like metrics.csv of "standard"
        run_metrics.to_csv(Path(dir.parent.parent / f"{cs_type}_all"))
        if post_stitch_delete:
            rmv_results = [os.remove(m) for m in metrics_paths]

def analyse_like_standard_metrics(metrics_paths: list[Path], prefix:str):
    """ 
    Equivalent func as `analyse_standard_metrics` except a prefix is used to 
    further denote the naming used for saved metrics in the same directory, 
    between GCS and RGCS. 

    This is kept until possible differentiating functionality is required, 
    otherwise TODO: Deprecate.
    """
    num_metrics = len(metrics_paths)

    # Sizes are unknown unless peeking is performed
    sum = np.zeros(0)
    sum_sq = np.zeros(0)
    header = [""]
    # Save initial path
    enumerated_metrics = enumerate(metrics_paths)

    for i, train_metrics_path in enumerated_metrics:
        with open(train_metrics_path) as f:
            cols = f.readline().split(',')
            ncols = len(cols)
            
        data = pd.read_csv(train_metrics_path.resolve(), usecols=range(0,ncols)).dropna()
        data = data.drop(columns=["step"])
        metrics = np.asarray(data.values)
        
        if i == 0:
            sum = np.zeros_like(metrics)
            sum_sq = np.zeros_like(metrics)
            header = list(data.columns)

        sum = np.sum([sum, metrics], axis=(0))
        sum_sq = np.sum([sum_sq, np.square(metrics)], axis=(0))

    global_mean = sum / num_metrics
    global_var = (sum_sq / num_metrics) - (np.square(global_mean))
    global_std = np.sqrt(global_var)
    dir = list(metrics_paths)[0].parent
    summary_metrics = [
        SummaryMetricInfo(
            values=global_std,
            save_path=dir / f"{prefix}_global_std",
            header=["global_std" + h for h in header]
        ),
        SummaryMetricInfo(
            values=global_mean,
            save_path= dir/ f"{prefix}_global_mean",
            header=["global_mean" + h for h in header]
        ),
    ]

    for i, summary_metric in enumerate(summary_metrics):
        print(f"Standard summary metric immediate file results and subdirs all saved into {summary_metric['save_path'].resolve()}")
        viz_sep_col(summary_metric)
        viz_all_col(summary_metric)

def analyse_cs_metrics(metrics_paths: list[Path], prefix:str) -> None:
    """ 
    Perform analysis steps of correlation store metrics. 
    """
    analyse_like_standard_metrics(metrics_paths, prefix)

In [ ]:
"""
Standard metric analysis helpers. 
"""

def analyse_standard_metrics(metrics_paths: list[Path]) -> None:
    """ 
    Perform analysis steps of standard metrics. 
    Assumptions:
    - Expects a list of by run SORTED ASC `metric_paths` that are across runs and point to a csv file. 
    - Resolving each path should yield a path to the metric file for one run as 1 metric per step in an epoch.  
    """
    num_metrics = len(metrics_paths)

    # Sizes are unknown unless peeking is performed
    sum = np.zeros(0)
    sum_sq = np.zeros(0)
    header = [""]
    # Save initial path
    enumerated_metrics = enumerate(metrics_paths)

    for i, train_metrics_path in enumerated_metrics:
        with open(train_metrics_path) as f:
            cols = f.readline().split(',')
            ncols = len(cols)
            
        data = pd.read_csv(train_metrics_path.resolve(), usecols=range(0,ncols)).dropna()
        data = data.drop(columns=["step"])
        metrics = np.asarray(data.values)
        
        if i == 0:
            sum = np.zeros_like(metrics)
            sum_sq = np.zeros_like(metrics)
            header = list(data.columns)

        sum = np.sum([sum, metrics], axis=(0))
        sum_sq = np.sum([sum_sq, np.square(metrics)], axis=(0))

    global_mean = sum / num_metrics
    global_var = (sum_sq / num_metrics) - (np.square(global_mean))
    global_std = np.sqrt(global_var)
    dir = list(metrics_paths)[0].parent
    summary_metrics = [
        SummaryMetricInfo(
            values=global_std,
            save_path=dir / "global_std",
            header=["global_std" + h for h in header]
        ),
        SummaryMetricInfo(
            values=global_mean,
            save_path= dir/ "global_mean",
            header=["global_mean" + h for h in header]
        ),
    ]

    for i, summary_metric in enumerate(summary_metrics):
        print(f"Standard summary metric immediate file results and subdirs all saved into {summary_metric['save_path'].resolve()}")
        viz_sep_col(summary_metric)
        viz_all_col(summary_metric)

In [ ]:
""" 
Main entry point for analysis. 
"""

outputs_dir = Path().resolve().parent / "outputs"
plt.switch_backend('WebAgg')
corr_store_names = dict(normed="GCS", raw="RGCS")

for experim_path in sorted(outputs_dir.iterdir()):
    version_dirs = sorted(experim_path.glob("multijob/v*"))
    for v_dir in version_dirs:
        # Get all csv file path matches to default logger file name for standard metrics
        train_metrics_paths = sorted(v_dir.rglob("run*/train/metrics.csv", case_sensitive=False))
        analyse_standard_metrics(train_metrics_paths)

        # Form equivalence to standard metrics that are one flat file for a run but with a 'step' column
        stitch_cs_metrics(v_dir, cs_type=corr_store_names["normed"])
        stitch_cs_metrics(v_dir, cs_type=corr_store_names["raw"])

        # Get all csv file path matches to custom name for both cs metrics
        gcs_paths = sorted(v_dir.rglob(f"run*/train/{corr_store_names["normed"]}*.csv", case_sensitive=False))
        rgcs_paths = sorted(v_dir.rglob(f"run*/train/{corr_store_names["raw"]}*.csv", case_sensitive=False))

        analyse_cs_metrics(gcs_paths, "gcs")
        analyse_cs_metrics(rgcs_paths, "rgcs")

plt.show()